In [1]:
# ==========================================================
# ACTIVIDAD 17. Elección del número de regímenes (k)
# Ajusta K-Means con k = 2 a 8 SOLO con empresas y bloque de
# entrenamiento, y lo evalúa también en el bloque de validación
# (mismas empresas, periodo nuevo). Las empresas reservadas
# NO se usan.
# ==========================================================
import time
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score

SEMILLA = 42
VALORES_K = range(2, 9)
MUESTRA_SILUETA = 20_000          # la silueta es lenta: se calcula sobre una muestra fija
COLUMNAS_Z = ["z_volatilidad_21", "z_cambio_volatilidad", "z_rendimiento_21",
              "z_caida_maximo_63", "z_volumen_anormal"]

drive.mount("/content/drive")
RAIZ = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Datos: empresas de entrenamiento en los bloques de entrenamiento y validación
# ----------------------------------------------------------
df = pd.read_parquet(RAIZ / "01_datos" / "variables_preparadas.parquet",
                     columns=["grupo", "bloque"] + COLUMNAS_Z)
df = df[df["grupo"] == "entrenamiento"]
X_ent = df.loc[df["bloque"] == "entrenamiento", COLUMNAS_Z].to_numpy()
X_val = df.loc[df["bloque"] == "validacion", COLUMNAS_Z].to_numpy()
print("Filas de entrenamiento:", len(X_ent), "| filas de validación:", len(X_val))

# Muestras fijas para la silueta (las mismas para todos los k)
rng = np.random.default_rng(SEMILLA)
idx_ent = rng.choice(len(X_ent), MUESTRA_SILUETA, replace=False)
idx_val = rng.choice(len(X_val), MUESTRA_SILUETA, replace=False)

# ----------------------------------------------------------
# 2. Un modelo por cada k
# ----------------------------------------------------------
filas = []
for k in VALORES_K:
    inicio = time.time()
    modelo = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_ent)
    etiquetas_ent = modelo.labels_
    etiquetas_val = modelo.predict(X_val)          # asigna el periodo nuevo SIN reentrenar

    # Estabilidad temporal: un K-Means entrenado desde cero en validación,
    # ¿agrupa las observaciones de la misma forma que el modelo de entrenamiento?
    modelo_val = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_val)
    filas.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silueta_ent": silhouette_score(X_ent[idx_ent], etiquetas_ent[idx_ent]),
        "silueta_val": silhouette_score(X_val[idx_val], etiquetas_val[idx_val]),
        "davies_bouldin_ent": davies_bouldin_score(X_ent, etiquetas_ent),
        "davies_bouldin_val": davies_bouldin_score(X_val, etiquetas_val),
        "ari_ent_vs_val": adjusted_rand_score(modelo_val.labels_, etiquetas_val),
        "grupo_min_pct_ent": np.bincount(etiquetas_ent).min() / len(etiquetas_ent) * 100,
        "grupo_min_pct_val": np.bincount(etiquetas_val, minlength=k).min() / len(etiquetas_val) * 100,
    })
    print(f"k = {k} listo en {time.time() - inicio:.0f} s")

resultados = pd.DataFrame(filas).set_index("k")
resultados["inercia_caida_pct"] = -resultados["inercia"].pct_change() * 100

# ----------------------------------------------------------
# 3. Resultados
# ----------------------------------------------------------
print("\n=== MÉTRICAS POR NÚMERO DE REGÍMENES ===")
print(resultados[["inercia", "inercia_caida_pct", "silueta_ent", "silueta_val",
                  "davies_bouldin_ent", "davies_bouldin_val", "ari_ent_vs_val",
                  "grupo_min_pct_ent", "grupo_min_pct_val"]])

resultados.to_csv(RAIZ / "02_modelos" / "seleccion_k.csv")
print("\nGuardado: 02_modelos/seleccion_k.csv")

Mounted at /content/drive
Filas de entrenamiento: 815829 | filas de validación: 488607
k = 2 listo en 21 s
k = 3 listo en 22 s
k = 4 listo en 24 s
k = 5 listo en 31 s
k = 6 listo en 35 s
k = 7 listo en 37 s
k = 8 listo en 42 s

=== MÉTRICAS POR NÚMERO DE REGÍMENES ===
        inercia  inercia_caida_pct  silueta_ent  silueta_val  davies_bouldin_ent  davies_bouldin_val  ari_ent_vs_val  grupo_min_pct_ent  grupo_min_pct_val
k                                                                                                                                                         
2 3,023,413.554                NaN        0.385        0.407               1.344               1.357           0.039             17.940             10.512
3 2,495,253.695             17.469        0.220        0.241               1.463               1.375           0.765             14.170              7.714
4 2,192,868.949             12.118        0.194        0.207               1.429               1.393           